# Notebook 01: basic data preparation (XLSX >>> cleaning >>> CSV)

Authors: Fabio Morea, Leyla Vesnic, Enrico Longato @ Area Science Park
 
Description: python scripts to clean and prepare data on regional companies.
This script imports data from 'IMPRESE.xlsx' and produces clean data in the form of .csv files.

License: 
scripts are available under CC-BY-4.0
data is not included in the package

Some description are in italian because of the nature of the data.


# comments and instructions for developers
The data are contained in two worksheets and are converted into two CSV files.



* creation of id_sede (un'impresa ha più sedi identificate da un numero 0 = SEDE , 1 = UL1...)

* creation of id_impresa (corrisponde a CF, ma 
                        è più efficiente 
                        non è "dato personale")

directory o file di input e output:
        ....

Preliminar operation:
    1- move the .xlsx in the working directory under the "input" folder



In [1]:
# Setup
import sys
import os
from pathlib import Path
import datetime
import pandas as pd
import numpy as np
import re
pd.options.display.max_columns = None

In [2]:
current_path = Path(os.getcwd())
data_subdir = "data/anagrafica"
data_path = current_path.parent / data_subdir
data_dir = str(data_path)
listafile = os.listdir(data_path)
listafile = list(filter(lambda x: 'imprese_fvg_' in x, listafile))  
assert len(listafile) >= 1, f"Error: can't find any file in {data_dir}"

In [3]:
data_path

WindowsPath('c:/Users/longato/OneDrive - Area Science Park/I2 - Evoluzione/FAIR/data/anagrafica')

In [4]:
# Auto-detect file_da_elaborare based on current month/year
import re
from datetime import datetime

# Get current date
today = datetime.today()
current_mm = f"{today.month:02d}"
current_yyyy = str(today.year)
current_date = (today.year, today.month)

# Find all available files matching the pattern
available_files = []
for fname in listafile:
    # Pattern: imprese_fvg_MM_YYYY.xlsx
    match = re.match(r'imprese_fvg_(\d{2})_(\d{4})\.xlsx', fname)
    if match:
        mm, yyyy = match.groups()
        file_date = (int(yyyy), int(mm))
        # Only include files not from the future
        if file_date <= current_date:
            available_files.append((file_date, f"{mm}_{yyyy}"))

if not available_files:
    # Fallback to latest available (even if future)
    for fname in listafile:
        match = re.match(r'imprese_fvg_(\d{2})_(\d{4})\.xlsx', fname)
        if match:
            mm, yyyy = match.groups()
            file_date = (int(yyyy), int(mm))
            available_files.append((file_date, f"{mm}_{yyyy}"))

# Sort by date descending and pick the most recent
available_files.sort(reverse=True)

if available_files:
    file_da_elaborare = available_files[0][1]
    print(f"Auto-detected: file_da_elaborare = '{file_da_elaborare}'")
    print(f"Current date: {current_mm}_{current_yyyy}")
    if available_files[0][0] == current_date:
        print(f"✓ File for current month available")
else:
    raise ValueError("No valid input files found matching pattern imprese_fvg_MM_YYYY.xlsx")

excel_file = data_dir + '\\imprese_fvg_' + file_da_elaborare + '.xlsx'
print(f'upload data in {excel_file}')

Auto-detected: file_da_elaborare = '02_2026'
Current date: 02_2026
✓ File for current month available
upload data in c:\Users\longato\OneDrive - Area Science Park\I2 - Evoluzione\FAIR\data\anagrafica\imprese_fvg_02_2026.xlsx


In [5]:
xl = pd.ExcelFile(  excel_file, engine="openpyxl")
xl_sheets = xl.sheet_names  # see all sheet names
assert xl_sheets == ['FRIULI Anagrafica', 'FRIULI codice attività']

C:\Users\longato\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\openpyxl\styles\stylesheet.py:226: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


## First sheet: FRIULI Anagrafica

In [6]:
df_anagrafica = xl.parse(   'FRIULI Anagrafica', 
                            header = 0, 
                            dtype=str,
                            keep_default_na=False)

C:\Users\longato\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\openpyxl\worksheet\_reader.py:223: UserWarning: Cell S53207 is marked as a date but the serial value 3032437.0 is outside the limits for dates. The cell will be treated as an error.
  warn(msg)
C:\Users\longato\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\openpyxl\worksheet\_reader.py:223: UserWarning: Cell S53208 is marked as a date but the serial value 3032437.0 is outside the limits for dates. The cell will be treated as an error.
  warn(msg)


In [7]:
# create a dictionary with the original and corrected column names
cols_df = pd.read_excel('cols_dict.xlsx', sheet_name='anagrafica') 
l1 = cols_df['nomi_colonne_originali']
l2 = cols_df['nomi_colonne_corretti']
cols_dic = dict(zip(l1,l2))
df_anagrafica.rename(columns=cols_dic, inplace=True)

In [8]:
#add colunms: source, mm_aaaa, type of location
df_anagrafica['fonte'] = 'I'
df_anagrafica['mm_aaaa'] = file_da_elaborare
df_anagrafica['n_sede'] = df_anagrafica['sede_ul'].str[3:] 

In [9]:
#convert 'E' in 'n_sede' to '0'
df_anagrafica.loc[ df_anagrafica['n_sede'] == 'E', 'n_sede'] = '0'
df_anagrafica['n_sede'].tolist()


['0',
 '0',
 '2',
 '4',
 '0',
 '0',
 '0',
 '0',
 '14',
 '2',
 '0',
 '1',
 '0',
 '1',
 '0',
 '0',
 '0',
 '0',
 '0',
 '1',
 '0',
 '0',
 '1',
 '0',
 '0',
 '1',
 '2',
 '3',
 '0',
 '0',
 '0',
 '0',
 '0',
 '0',
 '0',
 '1',
 '2',
 '3',
 '0',
 '4',
 '1',
 '2',
 '0',
 '0',
 '0',
 '2',
 '0',
 '1',
 '0',
 '0',
 '2',
 '0',
 '0',
 '0',
 '3',
 '5',
 '1',
 '8',
 '0',
 '0',
 '1',
 '0',
 '1',
 '0',
 '2',
 '4',
 '0',
 '0',
 '0',
 '0',
 '0',
 '2',
 '0',
 '0',
 '1',
 '0',
 '0',
 '0',
 '0',
 '0',
 '0',
 '0',
 '1',
 '0',
 '1',
 '1',
 '0',
 '0',
 '6',
 '0',
 '1',
 '18',
 '2',
 '11',
 '2',
 '0',
 '0',
 '0',
 '0',
 '2',
 '3',
 '0',
 '0',
 '0',
 '3',
 '0',
 '0',
 '0',
 '0',
 '1',
 '11',
 '2',
 '0',
 '0',
 '0',
 '4',
 '0',
 '2',
 '5',
 '0',
 '2',
 '1',
 '0',
 '2',
 '0',
 '1',
 '2',
 '0',
 '1',
 '1',
 '0',
 '0',
 '0',
 '0',
 '0',
 '3',
 '1',
 '0',
 '1',
 '2',
 '5',
 '0',
 '2',
 '6',
 '0',
 '0',
 '1',
 '4',
 '0',
 '0',
 '0',
 '1',
 '2',
 '3',
 '0',
 '1',
 '4',
 '21',
 '0',
 '12',
 '19',
 '20',
 '3',
 '0',
 '0',
 '

In [10]:
# info about the dataset
dim = df_anagrafica.shape 
print(f'dimension of the dataset (row,columns) = {dim}\n')
print(df_anagrafica.info())

dimension of the dataset (row,columns) = (178166, 58)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 178166 entries, 0 to 178165
Data columns (total 58 columns):
 #   Column                 Non-Null Count   Dtype 
---  ------                 --------------   ----- 
 0   cf                     178166 non-null  object
 1   prov                   178166 non-null  object
 2   reg_imp_n              178166 non-null  object
 3   rea                    178166 non-null  object
 4   sede_ul                178166 non-null  object
 5   n-albo_art             178166 non-null  object
 6   reg_imp_sez            178166 non-null  object
 7   ng2                    178166 non-null  object
 8   ng_esteso              178166 non-null  object
 9   tipo_impresa           178166 non-null  object
 10  data_isc_ri            178166 non-null  object
 11  data_isc_rd            178166 non-null  object
 12  data_isc_aa            178166 non-null  object
 13  data_apert_ul          178166 non-null  object
 1

In [11]:
#strip special characters
#chars_to_strip = '\\n\\t\\r|#'
pattern = r'[\n\t\r"\'|#\-*]|_x000D_'

cols_to_strip = ['denominazione', 'descrizione_attivita', 'indirizzo']

for col in cols_to_strip:
    df_anagrafica[col] = (
        df_anagrafica[col]
        .astype(str)
        .str.replace(pattern, ' ', regex=True)
        .str.strip()
    )
    print(col, "ok")

denominazione ok
descrizione_attivita ok
indirizzo ok


In [12]:
#keep only the active companies
df_anagrafica= df_anagrafica.loc[df_anagrafica['data_cess_att'].isin([''])]


In [13]:
# The dates in the original file have some issues, need to correct them with a function that: 
#   (1) subtract 3000 from the year, 
#   (2) properly handle errors and "nan", 
#   (3) remove dates earlier than 1800 or later than 2099

def anno_corretto(dstring:str) -> str:
    num =0
    dstring = str(dstring)[:10]
    if len(dstring) < 4:
        result = "NaT"
    else:
        try:
            num = int(dstring[0:2])
            if num >= 48 and num <= 51: #years between 1800 and 2100
                num = num - 30
                result = str(num) + dstring[2:]
            else:
                result = "NaT"
        except:
            result = "NaT"
    return result   

In [14]:
#test to verify the function anno_corretto
assert anno_corretto('x')   == 'NaT'
assert anno_corretto('1799-03-01') == 'NaT'
assert anno_corretto('2100-03-01') == 'NaT'
assert anno_corretto('4987-03-01') == '1987-03-01'
assert anno_corretto('5021-03-01') == '2021-03-01'
pass

In [15]:
# Normalize all date columns by applying anno_corretto() and converting them to datetime64.

cols_date = [d for d in df_anagrafica.columns if d.startswith('data')]
print(cols_date)
for col in cols_date:
    datestring3000 = df_anagrafica[col].tolist()
    datestring = [anno_corretto(item) for item in datestring3000]
    df_anagrafica[col] = pd.to_datetime(datestring)

['data_isc_ri', 'data_isc_rd', 'data_isc_aa', 'data_apert_ul', 'data_canc', 'data_ini_at', 'data_cess_att', 'data_fall', 'data_liquid', 'data_fine_aa', 'data_cost']


### id localiz e id impresa

In [16]:
# Create the key_cfl field to link id_localiz with the codes dataset,
# combining the tax code (cf) and the branch number (n_sede)

df_anagrafica['key_cfl'] = df_anagrafica['cf'] + '_'  + df_anagrafica['n_sede'].apply(str)
df_anagrafica.info()

<class 'pandas.core.frame.DataFrame'>
Index: 174996 entries, 0 to 178165
Data columns (total 59 columns):
 #   Column                 Non-Null Count   Dtype         
---  ------                 --------------   -----         
 0   cf                     174996 non-null  object        
 1   prov                   174996 non-null  object        
 2   reg_imp_n              174996 non-null  object        
 3   rea                    174996 non-null  object        
 4   sede_ul                174996 non-null  object        
 5   n-albo_art             174996 non-null  object        
 6   reg_imp_sez            174996 non-null  object        
 7   ng2                    174996 non-null  object        
 8   ng_esteso              174996 non-null  object        
 9   tipo_impresa           174996 non-null  object        
 10  data_isc_ri            173099 non-null  datetime64[ns]
 11  data_isc_rd            174979 non-null  datetime64[ns]
 12  data_isc_aa            31637 non-null   datetime6

In [17]:
# Create a temporary "years" column to compute and sort records by age:
# - calculate the earliest date across all date columns
# - compute the time difference from today
# - convert it into years
# - drop intermediate helper columns

df_anagrafica['min_date']   = pd.to_datetime( df_anagrafica[cols_date].min(axis=1) )
df_anagrafica['today']      = pd.to_datetime( pd.Timestamp.today().strftime('%Y-%m-%d') ) 
df_anagrafica['anni_timedelta'] =  (df_anagrafica['today']  - df_anagrafica['min_date'])
df_anagrafica['anni'] = df_anagrafica['anni_timedelta'].dt.total_seconds() / (60 * 60 * 24 * 365)
df_anagrafica.drop(columns='today')
df_anagrafica.drop(columns='anni_timedelta')

,cf,prov,reg_imp_n,rea,sede_ul,n-albo_art,reg_imp_sez,ng2,ng_esteso,tipo_impresa,data_isc_ri,data_isc_rd,data_isc_aa,data_apert_ul,data_canc,data_ini_at,data_cess_att,data_fall,data_liquid,denominazione,indirizzo,indirizzo_strad,indirizzo_cap,comune,indirizzo_fraz,indirizzo_altre,addetti_aaaa,addetti_indip,addetti_dip,piva,tel,capitale,descrizione_attivita,capitale_valuta,stato_impresa,tipo_sedeul_1,tipo_sedeul_2,tipo_sedeul_3,tipo_sedeul_4,tipo_sedeul_5,imp_sedi_ee,imp_eefvg,imp_pmi,imp_startup,imp_femmilile,imp_giovanile,imp_straniera,PEC,data_fine_aa,data_cost,tipo_localizzazione,last_update,prov_camera_commercio,REA_camera_commercio,PRO_LOCALIZZAZIONE,fonte,mm_aaaa,n_sede,key_cfl,min_date,today,anni
0,00000470310,GO,(GO007-1352),37843,SEDE,,O,SN,SN - SOCIETA' IN NOME COLLETTIVO,SOCIETA' DI PERSONE,1996-02-19,1975-01-14,NaT,NaT,NaT,NaT,NaT,NaT,NaT,PELLIZZARI SILVIO DI SEVERINO PELLIZZARI E C. ...,VIA PESCHERIA 4,,34071,CORMONS - GO,,,1999,0,0,00000470310,0481/60323,,COMMERCIO AL MINUTO TAB. MERC. I II (SOLTANT...,,INATTIVA,,,,,,,,NO,NO,NON FEMMINILE,NON GIOVANE,ITALIANA,,NaT,1974-08-26,SE - SEDE PRINCIPALE,2022-08-01 00:00:00,GO,37843,0,I,02_2026,0,00000470310_0,1974-08-26,2026-02-24,51.534247
1,00002070324,TS,(TS006-7084),65026,SEDE,,O,SR,SR - SOCIETA' A RESPONSABILITA' LIMITATA,SOCIETA' DI CAPITALE,1996-02-19,1969-01-30,NaT,NaT,NaT,1969-01-30,NaT,NaT,NaT,B.F.B. CASA DI SPEDIZIONI S.R.L.,VIA CORTI 2,,34123,TRIESTE - TS,,,2025,0,14,00002070324,040/3220798,20000,"SPEDIZIONI DOGANALI, DAL 30/01/1969",EURO,ATTIVA,,,,,,,,NO,NO,NON FEMMINILE,NON GIOVANE,ITALIANA,CLAUDIO.BROSCH@PEC.BFBTRIESTE.COM,NaT,1969-01-30,SE - SEDE PRINCIPALE,2025-05-22 00:00:00,TS,65026,0,I,02_2026,0,00002070324_0,1969-01-30,2026-02-24,57.106849
2,00002070324,TS,(TS006-7084),65026,UL-2,,O,SR,SR - SOCIETA' A RESPONSABILITA' LIMITATA,SOCIETA' DI CAPITALE,1996-02-19,1969-01-30,NaT,2007-08-01,NaT,NaT,NaT,NaT,NaT,B.F.B. CASA DI SPEDIZIONI S.R.L.,FERNETTI 5,,34016,MONRUPINO - TS,,"AUTOPORTO, MODULO 50",2025,0,14,00002070324,040/3220798,20000,"SPEDIZIONI DOGANALI, DAL 30/01/1969",EURO,ATTIVA,U - UFFICIO,,,,,,,NO,NO,NON FEMMINILE,NON GIOVANE,ITALIANA,CLAUDIO.BROSCH@PEC.BFBTRIESTE.COM,NaT,1969-01-30,UL - UNITÀ LOCALE,2025-05-22 00:00:00,TS,65026,2,I,02_2026,2,00002070324_2,1969-01-30,2026-02-24,57.106849
3,00002070324,TS,(TS006-7084),65026,UL-4,,O,SR,SR - SOCIETA' A RESPONSABILITA' LIMITATA,SOCIETA' DI CAPITALE,1996-02-19,1969-01-30,NaT,2015-10-15,NaT,NaT,NaT,NaT,NaT,B.F.B. CASA DI SPEDIZIONI S.R.L.,PUNTO FRANCO NUOVO EX CULP SN,,34123,TRIESTE - TS,,,2025,0,14,00002070324,040/3220798,20000,"SPEDIZIONI DOGANALI, DAL 30/01/1969",EURO,ATTIVA,U - UFFICIO,,,,,,,NO,NO,NON FEMMINILE,NON GIOVANE,ITALIANA,CLAUDIO.BROSCH@PEC.BFBTRIESTE.COM,NaT,1969-01-30,UL - UNITÀ LOCALE,2025-05-22 00:00:00,TS,65026,4,I,02_2026,4,00002070324_4,1969-01-30,2026-02-24,57.106849
4,00006350938,PN,(PN033-1275),9764,SEDE,,O,AS,AS - SOCIETA' IN ACCOMANDITA SEMPLICE,SOCIETA' DI PERSONE,1996-02-19,1965-11-18,NaT,NaT,NaT,1965-10-30,NaT,NaT,NaT,CARROZZERIA AZZANESE DI LINO MAGGIOLO S.A.S.,VIA FIUMICINO 14,,33082,AZZANO DECIMO - PN,,,2025,1,0,00006350938,0434/631194,0,DAL 09/07/2013 COMMERCIO AL DETTAGLIO DI CARBU...,EURO,ATTIVA,,,,,,,,NO,NO,NON FEMMINILE,NON GIOVANE,ITALIANA,CAR.AZZANESESNC@TICERTIFICA.IT,NaT,1965-10-30,SE - SEDE PRINCIPALE,2025-05-12 00:00:00,PN,9764,0,I,02_2026,0,00006350938_0,1965-10-30,2026-02-24,60.361644
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
178161,ZZZNTN78H14L195O,UD,(UD-2009-7369),270774,SEDE,UD-87280,A-P,DI,DI - IMPRESA INDIVIDUALE,IMPRESE INDIVIDUALI,2009-03-10,2009-03-10,2009-03-10,NaT,NaT,2009-03-10,NaT,NaT,NaT,ZOZZOLI ANTONIO,VIA AONESI 42,,33027,PAULARO - UD,,,2025,1,0,02564170302,328/8878256,,"ATTIVITA : COSTRUZIONE, RISTRUTTURAZIONE DI ED...",,ATTIVA,,,,,,,,NO

In [18]:
# Sort the dataframe by the computed age in years and the composite key (key_cfl), in descending order
df_anagrafica.sort_values(by = ['anni', 'key_cfl'], ascending=False, inplace = True)

In [19]:
# Create id_localiz from the dataframe index:
# - reset the index to expose it as a column
# - reset again to generate a sequential numeric id
# - assign id_localiz as index + 1
# - restore the original ordering

df_anagrafica.reset_index(inplace=True)
df_anagrafica.reset_index(inplace=True)
df_anagrafica['id_localiz'] = df_anagrafica['index'] + 1

df_anagrafica.sort_values(by = 'index', inplace = True)

In [20]:
# Create a unique company identifier linked to the tax code (cf):
# - select tax code and company name
# - drop duplicate tax codes
# - reset the index to generate a sequential id
# - assign id_impresa starting from 1

df_cf_univoco = df_anagrafica[['cf', 'denominazione']]
df_cf_univoco.drop_duplicates(subset='cf', inplace = True)
df_cf_univoco.reset_index(inplace = True)
df_cf_univoco.reset_index(inplace = True)
df_cf_univoco['id_impresa'] = df_cf_univoco['level_0'] +1
df_cf_univoco.columns

C:\Users\longato\AppData\Local\Temp\ipykernel_24208\1152268165.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_cf_univoco.drop_duplicates(subset='cf', inplace = True)
C:\Users\longato\AppData\Local\Temp\ipykernel_24208\1152268165.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_cf_univoco['id_impresa'] = df_cf_univoco['level_0'] +1


Index(['level_0', 'index', 'cf', 'denominazione', 'id_impresa'], dtype='object')

In [21]:
# Drop intermediate index columns and reorder the dataframe to show id_impresa and cf
df_cf_univoco.drop(columns=['level_0', 'index'], inplace = True)
cols_order = ['id_impresa', 'cf' ]
df_cf_univoco = df_cf_univoco[cols_order]

C:\Users\longato\AppData\Local\Temp\ipykernel_24208\60958292.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_cf_univoco.drop(columns=['level_0', 'index'], inplace = True)


In [22]:
df_anagrafica.shape[0] #conta righe

174996

In [23]:
# Join the main anagraphic dataset with the deduplicated company table using the tax code (cf)

df_anagrafica = df_anagrafica.merge(df_cf_univoco, on = 'cf', how = 'inner') 

### Checks on local units without reference headquarters – duplicate detection – date validation


In [24]:
# Function to create a binary flag column based on whether the unit is a headquarters or a local unit

def f(row):
    if row['sede_ul'] == 'SEDE':
        val = 's'
    else:
        val = 'u'
    return val

In [25]:
# new column 
df_anagrafica['SEDE_UL_num'] = df_anagrafica.apply(f, axis = 1)

In [26]:
# Build a pivot table to count headquarters vs local units for each tax code (cf)

pivot_test = df_anagrafica.pivot_table(index = "cf",
                                values = "sede_ul",
                                columns='SEDE_UL_num',
                                aggfunc = lambda x : x.count(),
                                margins = True,
                                fill_value=0)
pivot_test

SEDE_UL_num,s,u,All
cf,,,
00000470310,1,0,1
00002070324,1,2,3
00006350938,1,0,1
00006500938,1,0,1
00006550313,1,0,1
...,...,...,...
ZZZRRT70L67L424I,1,0,1
ZZZRRT74D06Z700T,1,0,1
ZZZSVN46E60G381I,1,0,1


In [27]:
## Extract the list cf with no headquarters (s == 0)

df_test = pivot_test
cf_no_sede = df_test[ df_test['s']==0 ]
cf_no_sede = cf_no_sede.reset_index()
cf_no_sede = cf_no_sede['cf'].to_list()


In [28]:
# Remove records whose CF has no headquarters

df_anagrafica = df_anagrafica[~df_anagrafica['cf'].isin(cf_no_sede)]

### Duplicate management

In [29]:
# Identify duplicate records based on CF and sede_ul
df_dup = df_anagrafica[df_anagrafica.duplicated(subset=['cf','sede_ul'], keep=False)]

In [30]:
# Keep only duplicates where the province differs from the Chamber of Commerce province,
# then inspect records for a specific CF
df_dup = df_dup[df_dup['prov'] != df_dup['prov_camera_commercio']]
df_dup[df_dup['cf'] == '07381630966']

,level_0,index,cf,prov,reg_imp_n,rea,sede_ul,n-albo_art,reg_imp_sez,ng2,ng_esteso,tipo_impresa,data_isc_ri,data_isc_rd,data_isc_aa,data_apert_ul,data_canc,data_ini_at,data_cess_att,data_fall,data_liquid,denominazione,indirizzo,indirizzo_strad,indirizzo_cap,comune,indirizzo_fraz,indirizzo_altre,addetti_aaaa,addetti_indip,addetti_dip,piva,tel,capitale,descrizione_attivita,capitale_valuta,stato_impresa,tipo_sedeul_1,tipo_sedeul_2,tipo_sedeul_3,tipo_sedeul_4,tipo_sedeul_5,imp_sedi_ee,imp_eefvg,imp_pmi,imp_startup,imp_femmilile,imp_giovanile,imp_straniera,PEC,data_fine_aa,data_cost,tipo_localizzazione,last_update,prov_camera_commercio,REA_camera_commercio,PRO_LOCALIZZAZIONE,fonte,mm_aaaa,n_sede,key_cfl,min_date,today,anni_timedelta,anni,id_localiz,id_impresa,SEDE_UL_num
99616,108017,102339,07381630966,UD,(MI-2011-59418),1954900,UL-4,,O,SR,SR - SOCIETA' A RESPONSABILITA' LIMITATA,SOCIETA' DI CAPITALE,2011-03-22,2011-03-22,NaT,2014-04-28,NaT,NaT,NaT,NaT,NaT,GIOIA GIOCHI S.R.L.,VIA PALMANOVA 87,,33100,UDINE - UD,,,2025,0,11,07381630966,,20000,SALA GIOCHI,EURO,ATTIVA,SAD - SALA GIOCHI,,,,,,,NO,NO,FORTE,NON GIOVANE,ITALIANA,GIOIAGIOCHI@CERTAPEC.IT,NaT,2011-03-16,UL - UNITÀ LOCALE,2025-12-22 00:00:00,MI,1954900,4,I,02_2026,4,07381630966_4,2011-03-16,2026-02-24,5459 days,14.956164,102340,46014,u
99618,108015,102341,07381630966,UD,(MI-2011-59418),1954900,UL-5,,O,SR,SR - SOCIETA' A RESPONSABILITA' LIMITATA,SOCIETA' DI CAPITALE,2011-03-22,2011-03-22,NaT,2015-10-17,NaT,NaT,NaT,NaT,NaT,GIOIA GIOCHI S.R.L.,VIALE PALMANOVA 89,,33100,UDINE - UD,,,2025,0,11,07381630966,,20000,SALA GIOCHI,EURO,ATTIVA,BAR - BAR,,,,,,,NO,NO,FORTE,NON GIOVANE,ITALIANA,GIOIAGIOCHI@CERTAPEC.IT,NaT,2011-03-16,UL - UNITÀ LOCALE,2025-12-22 00:00:00,MI,1954900,5,I,02_2026,5,07381630966_5,2011-03-16,2026-02-24,5459 days,14.956164,102342,46014,u


In [31]:
# Drop duplicated records identified for removal using their row indices
id_df_dup = df_dup.index.values.tolist()
df_anagrafica.drop(id_df_dup, axis=0, inplace=True)

In [32]:
# Recheck for remaining duplicates based on CF and sede_ul after cleanup
df_dup_test_final = df_anagrafica[df_anagrafica.duplicated(subset=['cf','sede_ul'], keep=False)]
df_dup_test_final

,level_0,index,cf,prov,reg_imp_n,rea,sede_ul,n-albo_art,reg_imp_sez,ng2,ng_esteso,tipo_impresa,data_isc_ri,data_isc_rd,data_isc_aa,data_apert_ul,data_canc,data_ini_at,data_cess_att,data_fall,data_liquid,denominazione,indirizzo,indirizzo_strad,indirizzo_cap,comune,indirizzo_fraz,indirizzo_altre,addetti_aaaa,addetti_indip,addetti_dip,piva,tel,capitale,descrizione_attivita,capitale_valuta,stato_impresa,tipo_sedeul_1,tipo_sedeul_2,tipo_sedeul_3,tipo_sedeul_4,tipo_sedeul_5,imp_sedi_ee,imp_eefvg,imp_pmi,imp_startup,imp_femmilile,imp_giovanile,imp_straniera,PEC,data_fine_aa,data_cost,tipo_localizzazione,last_update,prov_camera_commercio,REA_camera_commercio,PRO_LOCALIZZAZIONE,fonte,mm_aaaa,n_sede,key_cfl,min_date,today,anni_timedelta,anni,id_localiz,id_impresa,SEDE_UL_num
90,2875,92,00041700386,FE,(FE008-266),21210,UL-2,,O,SC,SC - SOCIETA' COOPERATIVA,ALTRE FORME,1996-02-19,1940-06-22,NaT,NaT,NaT,NaT,NaT,2024-04-17,NaT,COOPERATIVA MURATORI RIUNITI SOC. COOP. A R....,VIA DELL ARTIGIANATO 239,,44026,MESOLA - FE,,,2021,0,0,00041700386,0532/853411,,"ESERCIZIO DI LAVORI MURARI, STRADALI, FLUVIALI...",,LIQUIDAZIONE,,,,,,,,NO,NO,NON FEMMINILE,NON GIOVANE,ITALIANA,F19.2011FERRARA@PECFALLIMENTI.IT,NaT,1939-04-03,UL - UNITÀ LOCALE,2024-10-03 00:00:00,FE,21210,2,I,02_2026,2,00041700386_2,1939-04-03,2026-02-24,31739 days,86.956164,93,56,u
92,2876,94,00041700386,PN,(FE008-266),21210,UL-2,,O,SC,SC - SOCIETA' COOPERATIVA,ALTRE FORME,1996-02-19,1940-06-22,NaT,1992-01-20,NaT,NaT,NaT,2024-04-17,NaT,COOPERATIVA MURATORI RIUNITI SOC. COOP. A R....,VIALE DELLA REPUBBLICA 66,,33080,FIUME VENETO - PN,,,2021,0,0,00041700386,0532/853411,,"ESERCIZIO DI LAVORI MURARI, STRADALI, FLUVIALI...",,ATTIVA,UA - UFFICIO AMMINISTRATIVO,UT - UFFICIO TECNICO,,,,,,NO,NO,NON FEMMINILE,NON GIOVANE,ITALIANA,F19.2011FERRARA@PECFALLIMENTI.IT,NaT,1939-04-03,UL - UNITÀ LOCALE,2024-10-03 00:00:00,PN,48861,2,I,02_2026,2,00041700386_2,1939-04-03,2026-02-24,31739 days,86.956164,95,56,u
238,16413,247,00050830314,GO,(TV-2003-39672),301147,UL-1,,G-O,SN,SN - SOCIETA' IN NOME COLLETTIVO,SOCIETA' DI PERSONE,2003-08-06,2003-08-05,NaT,1980-12-22,NaT,NaT,NaT,NaT,NaT,SOCIETA AGRICOLA TURCO DI ONGARO ASSUNTA & C....,LOCALITA DOSSI BOSCAT,,34073,GRADO - GO,,,2025,0,1,01001730306,,0,ATTIVITA : SVOLTA PRESSO L UNITA LOCALE.,EURO,ATTIVA,AA - AZIENDA AGRICOLA,,,,,,,NO,NO,NON FEMMINILE,NON GIOVANE,ITALIANA,AGRICOLA.TURCO@PEC.IT,NaT,1980-12-22,UL - UNITÀ LOCALE,2021-07-14 00:00:00,GO,61448,1,I,02_2026,1,00050830314_1,1980-12-22,2026-02-24,16500 days,45.205479,248,136,u
241,16415,250,00050830314,UD,(TV-2003-39672),301147,UL-1,,G-O,SN,SN - SOCIETA' IN NOME COLLETTIVO,SOCIETA' DI PERSONE,2003-08-06,2003-08-05,NaT,2018-12-17,NaT,NaT,NaT,NaT,NaT,SOCIETA AGRICOLA TURCO DI ONGARO ASSUNTA & C....,VIA TALMASSONS 6,,33061,RIVIGNANO TEOR - UD,,,2025,0,1,01001730306,,0,ATTIVITA : SVOLTA PRESSO L UNITA LOCALE.,EURO,ATTIVA,AA - AZIENDA AGRICOLA,,,,,,,NO,NO,NON FEMMINILE,NON GIOVANE,ITALIANA,AGRICOLA.TURCO@PEC.IT,NaT,1980-12-22,UL - UNITÀ LOCALE,2021-07-14 00:00:00,UD,350772,1,I,02_2026,1,00050830314_1,1980-12-22,2026-02-24,16500 days,45.205479,251,136,u
742,1437,756,00053810149,PN,(SO061-1),3167,UL-1,,O,SP,SP - SOCIETA' PER AZIONI,SOCIETA' DI CAPITALE,1996-02-19,1925-11-22,NaT,2024-11-04,NaT,NaT,NaT,NaT,NaT,BANCA POPOLARE DI SONDRIO SOCIETA PER AZIONI,VIALE GUGLIELMO MARCONI 28,,33170,PORDENONE - PN,,,2025,0,3180,00053810149,342/528111,1360157331,ATTIVITA BANCARIA.,EURO,ATTIVA,AG - AGENZIA,,,,,,,NO,NO,NON FEMMINILE,NON GIOVANE,ITALIANA,POSTACERTIFICATA@PEC.POPSO.IT,NaT,1871-03-04,UL - UNITÀ LOCALE,2025-12-19 00:00:00,PN,374532,1,I,02_2026,1,00053810149_1,1871-03-04,2026-02-24,56605 days,155.082192,757,195,u
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
170015,148042,173162,TSSLSS83P41L407V

In [33]:
# Check whether any remaining duplicates refer to the same local unit in the same province
df_dup_test_final_2 = df_dup_test_final[df_dup_test_final.duplicated(subset=['cf','sede_ul','prov'], keep=False)]
df_dup_test_final_2

,level_0,index,cf,prov,reg_imp_n,rea,sede_ul,n-albo_art,reg_imp_sez,ng2,ng_esteso,tipo_impresa,data_isc_ri,data_isc_rd,data_isc_aa,data_apert_ul,data_canc,data_ini_at,data_cess_att,data_fall,data_liquid,denominazione,indirizzo,indirizzo_strad,indirizzo_cap,comune,indirizzo_fraz,indirizzo_altre,addetti_aaaa,addetti_indip,addetti_dip,piva,tel,capitale,descrizione_attivita,capitale_valuta,stato_impresa,tipo_sedeul_1,tipo_sedeul_2,tipo_sedeul_3,tipo_sedeul_4,tipo_sedeul_5,imp_sedi_ee,imp_eefvg,imp_pmi,imp_startup,imp_femmilile,imp_giovanile,imp_straniera,PEC,data_fine_aa,data_cost,tipo_localizzazione,last_update,prov_camera_commercio,REA_camera_commercio,PRO_LOCALIZZAZIONE,fonte,mm_aaaa,n_sede,key_cfl,min_date,today,anni_timedelta,anni,id_localiz,id_impresa,SEDE_UL_num
94851,167911,97408,04868550270,UD,(VE-2024-24526),458649,UL-1,VE-104023,A-O,SR,SR - SOCIETA' A RESPONSABILITA' LIMITATA,SOCIETA' DI CAPITALE,2024-07-05,2024-07-04,2024-11-15,2024-11-15,NaT,NaT,NaT,NaT,NaT,EDICAR SERVICE S.R.L.,VIA MAGRINI 2,,33050,PRECENICCO - UD,,,2025,1,2,04868550270,,20000,OFFICINA MECCANICA IN GENERE SEZIONE MECCATR...,EURO,ATTIVA,DEP - DEPOSITO,,,,,,,NO,NO,NON FEMMINILE,NON GIOVANE,ITALIANA,EDICARPEC@PEC.IT,NaT,2024-07-03,UL - UNITÀ LOCALE,2025-12-05 00:00:00,UD,374492,1,I,02_2026,1,04868550270_1,2024-07-03,2026-02-24,601 days,1.646575,97409,45345,u
94852,167912,97409,04868550270,UD,(VE-2024-24526),458649,UL-1,VE-126949,A-O,SR,SR - SOCIETA' A RESPONSABILITA' LIMITATA,SOCIETA' DI CAPITALE,2024-07-05,2024-07-04,2025-04-29,2024-11-15,NaT,NaT,NaT,NaT,NaT,EDICAR SERVICE S.R.L.,VIA MAGRINI 2,,33050,PRECENICCO - UD,,,2025,1,2,04868550270,,20000,OFFICINA MECCANICA IN GENERE SEZIONE MECCATR...,EURO,ATTIVA,DEP - DEPOSITO,,,,,,,NO,NO,NON FEMMINILE,NON GIOVANE,ITALIANA,EDICARPEC@PEC.IT,NaT,2024-07-03,UL - UNITÀ LOCALE,2025-12-05 00:00:00,UD,374492,1,I,02_2026,1,04868550270_1,2024-07-03,2026-02-24,601 days,1.646575,97410,45345,u
94855,167913,97412,04868550270,VE,(VE-2024-24526),458649,SEDE,VE-104023,A-O,SR,SR - SOCIETA' A RESPONSABILITA' LIMITATA,SOCIETA' DI CAPITALE,2024-07-05,2024-07-04,2024-11-15,NaT,NaT,2024-11-15,NaT,NaT,NaT,EDICAR SERVICE S.R.L.,VIA ALDO MORO 1,,30025,FOSSALTA DI PORTOGRUARO - VE,,,2025,1,2,04868550270,,20000,OFFICINA MECCANICA IN GENERE SEZIONE MECCATR...,EURO,ATTIVA,,,,,,,,NO,NO,NON FEMMINILE,NON GIOVANE,ITALIANA,EDICARPEC@PEC.IT,NaT,2024-07-03,SE - SEDE PRINCIPALE,2025-12-05 00:00:00,VE,458649,0,I,02_2026,0,04868550270_0,2024-07-03,2026-02-24,601 days,1.646575,97413,45345,s
94856,167914,97413,04868550270,VE,(VE-2024-24526),458649,SEDE,VE-126949,A-O,SR,SR - SOCIETA' A RESPONSABILITA' LIMITATA,SOCIETA' DI CAPITALE,2024-07-05,2024-07-04,2025-04-29,NaT,NaT,2024-11-15,NaT,NaT,NaT,EDICAR SERVICE S.R.L.,VIA ALDO MORO 1,,30025,FOSSALTA DI PORTOGRUARO - VE,,,2025,1,2,04868550270,,20000,OFFICINA MECCANICA IN GENERE SEZIONE MECCATR...,EURO,ATTIVA,,,,,,,,NO,NO,NON FEMMINILE,NON GIOVANE,ITALIANA,EDICARPEC@PEC.IT,NaT,2024-07-03,SE - SEDE PRINCIPALE,2025-12-05 00:00:00,VE,458649,0,I,02_2026,0,04868550270_0,2024-07-03,2026-02-24,601 days,1.646575,97414,45345,s


## Second sheet: FRIULI activity codes


In [34]:
# Load the "FRIULI codice attività" sheet into a dataframe, forcing all columns to string
df_codici = xl.parse('FRIULI codice attività',  
                    header = 0,
                    dtype=str,
                    keep_default_na=False) 

In [35]:
# Create a dictionary mapping original column names to corrected ones,
# then rename the dataframe columns accordingly

cols_df = pd.read_excel('cols_dict.xlsx', sheet_name='codici') 
l1 = cols_df['nomi_colonne_originali']
print(l1)
l2 = cols_df['nomi_colonne_corretti']
print(l2)

cols_dic = dict(zip(l1,l2))

print(cols_dic)

df_codici.rename(columns=cols_dic, inplace=True)

df_codici


0                  c fiscale
1            SIGLA PROVINCIA
2                        rea
3                        loc
4                    imp att
5                 ateco 2007
6     descrizione ateco 2007
7                        NaN
8                        NaN
9                        NaN
10                       NaN
Name: nomi_colonne_originali, dtype: object
0             cf
1           prov
2            rea
3          loc_n
4     ateco_tipo
5          ateco
6     ateco_desc
7          fonte
8        mm_aaaa
9     id_impresa
10    id_localiz
Name: nomi_colonne_corretti, dtype: object
{'c fiscale': 'cf', 'SIGLA PROVINCIA': 'prov', 'rea': 'rea', 'loc': 'loc_n', 'imp att': 'ateco_tipo', 'ateco 2007': 'ateco', 'descrizione ateco 2007': 'ateco_desc', nan: 'id_localiz'}


,cf,prov,rea,loc_n,ateco_tipo,ateco,ateco_desc
0,00000470310,GO,37843,0,,,
1,00002070324,TS,65026,0,I,52.26.0,Altre attività di supporto ai trasporti
2,00002070324,TS,65026,0,I,52.29.1,Spedizionieri e agenzie di operazioni doganali
3,00002070324,TS,65026,0,P,52.26.0,Altre attività di supporto ai trasporti
4,00002070324,TS,65026,0,P,52.29.1,Spedizionieri e agenzie di operazioni doganali
...,...,...,...,...,...,...,...
673586,ZZZSVN46E60G381I,UD,227639,0,P,01.41.00,Allevamento di bovini da latte
673587,ZZZTMS74S21L483O,UD,235833,0,I,01.21,Coltivazione di uva
673588,ZZZTMS74S21L483O,UD,235833,0,I,01.21.00,Coltivazione di uva
673589,ZZZTMS74S21L483O,UD,235833,0,P,01.21,Coltivazione di uva


In [36]:
# Add source identifier and reference month/year fields
df_codici['fonte'] = 'I'
df_codici['mm_aaaa'] = file_da_elaborare

In [37]:
# Create key_cfl in df_codici to link to id_localiz,
# then extract the mapping table from df_anagrafica

df_codici['key_cfl'] = df_codici['cf'] + '_'  + df_codici['loc_n']
df_temp = df_anagrafica[['key_cfl',  'id_impresa', 'id_localiz']]


In [38]:
# Add id_localiz to df_codici by joining on key_cfl
df_codici = df_codici.merge(df_temp, on = 'key_cfl', how = 'inner') 


In [39]:
# Filter the codes sheet as well by removing CF values with no headquarters
df_codici = df_codici[~df_codici['cf'].isin(cf_no_sede)]

In [40]:
# Identify duplicate records in df_codici based on CF and location/activity fields
df_cod_dup = df_codici[df_codici.duplicated(subset=['cf','prov', 'rea', 'loc_n', 'ateco_tipo','ateco','ateco_desc'], keep=False)]


In [41]:
df_codici.shape[0]

670045

### Write output CSV files for multiple targets


#### File .csv per Repository

Since the "tipo_sedeul_n" fields need to be merged, I create a copy of the dataframe so that the merge can be performed on the copy and saved in the "Repository" version.


In [42]:
# Create a copy of the dataframe for the repository version,
# merge the tipo_sedeul_* fields into a single column,
# and drop the original component columns
df_anagrafica_repo = df_anagrafica.copy()
df_anagrafica_repo['tipo_sedeul'] =  df_anagrafica_repo.tipo_sedeul_1+df_anagrafica_repo.tipo_sedeul_2+df_anagrafica_repo.tipo_sedeul_3+df_anagrafica_repo.tipo_sedeul_4+df_anagrafica_repo.tipo_sedeul_5
df_anagrafica_repo.drop(columns=['tipo_sedeul_1', 'tipo_sedeul_2', 'tipo_sedeul_3', 'tipo_sedeul_4',
       'tipo_sedeul_5'], inplace=True)

In [43]:
# Save the ANAGRAFICA file in the "repository" version:
# - load the export column mapping and order
# - sort columns according to the repository layout
# - export the selected fields to a pipe-delimited CSV file
file_risultati = data_dir + '\\' + 'imprese_anagrafica.csv'
cols_df = pd.read_excel('cols_dict.xlsx', sheet_name='anagrafica',
            usecols = ['nomi_colonne_export','ordine_repo']).dropna() 
cols_df.sort_values('ordine_repo', inplace = True)
cols_to_use = list(cols_df['nomi_colonne_export'])
df_anagrafica_repo[cols_to_use].to_csv(  file_risultati, 
                                    sep ='|',   
                                    encoding='utf-8-sig', 
                                    index=False)

In [44]:
# Save the CODICI file in the "repository" version:
# - load the export column mapping and order
# - sort columns according to the repository layout
# - export the selected fields to a pipe-delimited CSV file
file_risultati = data_dir + '\\' + 'imprese_codici.csv'
cols_df = pd.read_excel('cols_dict.xlsx', sheet_name='codici',
            usecols = ['nomi_colonne_corretti','ordine_repo']).dropna() 
cols_df.sort_values('ordine_repo', inplace = True)
cols_to_use = list(cols_df['nomi_colonne_corretti'])
df_codici[cols_to_use].to_csv(      file_risultati, 
                                    sep ='|',   
                                    encoding='utf-8-sig', 
                                    index=False)

#### CSV file for Innovation Intelligence (local units including those outside FVG)


In [45]:
# Save the ANAGRAFICA file in the "Innovation Intelligence" version:
# - load the export column mapping and order
# - sort columns according to the Innovation Intelligence layout
# - export the selected fields to a pipe-delimited CSV file
file_risultati = data_dir + '\\' + 'iifvg_anagrafica.csv'
cols_df = pd.read_excel('cols_dict.xlsx', sheet_name='anagrafica',
            usecols = ['nomi_colonne_corretti','ordine_iifvg']).dropna() 
cols_df.sort_values('ordine_iifvg', inplace = True)
cols_to_use = list(cols_df['nomi_colonne_corretti'])
df_anagrafica[cols_to_use].to_csv(  file_risultati, 
                                    sep ='|',   
                                    encoding='utf-8-sig', 
                                    index=False)

In [46]:
# Save the CODICI file in the "Innovation Intelligence" version:
# - load the export column mapping and order
# - sort columns according to the Innovation Intelligence layout
# - export the selected fields to a pipe-delimited CSV file
file_risultati = data_dir + '\\' + 'iifvg_codici.csv'
cols_df = pd.read_excel('cols_dict.xlsx', sheet_name='codici',
            usecols = ['nomi_colonne_corretti','ordine_iifvg']).dropna() 
cols_df.sort_values('ordine_iifvg', inplace = True)
cols_to_use = list(cols_df['nomi_colonne_corretti'])
df_codici[cols_to_use].to_csv(      file_risultati, 
                                    sep ='|',   
                                    encoding='utf-8-sig', 
                                    index=False)

#### File .csv per Innovation Intelligence (solamente unità locali FVG)

In [47]:
# Filter out local units outside the FVG provinces from the ANAGRAFICA dataset:
# - keep headquarters
# - remove non-FVG local units
local_sede = ["SEDE"]
prov_FVG = ["GO", "PN", "UD", "TS"]
unità_locali_extraFVG_filter = ~df_anagrafica["sede_ul"].isin(local_sede) & ~df_anagrafica["prov"].isin(prov_FVG)
df_anagrafica = df_anagrafica[~unità_locali_extraFVG_filter]

In [48]:
# Save the ANAGRAFICA file in the "Innovation Intelligence" version
# after removing local units outside the FVG region
file_risultati = data_dir + '\\' + 'iifvg_anagrafica_filtrato.csv'
cols_df = pd.read_excel('cols_dict.xlsx', sheet_name='anagrafica',
            usecols = ['nomi_colonne_corretti','ordine_iifvg']).dropna() 
cols_df.sort_values('ordine_iifvg', inplace = True)
cols_to_use = list(cols_df['nomi_colonne_corretti'])
df_anagrafica[cols_to_use].to_csv(  file_risultati, 
                                    sep ='|',   
                                    encoding='utf-8-sig', 
                                    index=False)

In [49]:
# Filter out local units outside the FVG provinces from the CODICI dataset:
# - keep headquarters (loc_n == "0")
# - remove non-FVG local units
local_sede = ["0"]
prov_FVG = ["GO", "PN", "UD", "TS"]
unità_locali_extraFVG_filter = ~df_codici["loc_n"].isin(local_sede) & ~df_codici["prov"].isin(prov_FVG)
df_codici = df_codici[~unità_locali_extraFVG_filter]

In [50]:
# Save the CODICI file in the "Innovation Intelligence" version
# after removing local units outside the FVG region
file_risultati = data_dir + '\\' + 'iifvg_codici_filtrato.csv'
cols_df = pd.read_excel('cols_dict.xlsx', sheet_name='codici',
            usecols = ['nomi_colonne_corretti','ordine_iifvg']).dropna() 
cols_df.sort_values('ordine_iifvg', inplace = True)
cols_to_use = list(cols_df['nomi_colonne_corretti'])
df_codici[cols_to_use].to_csv(      file_risultati, 
                                    sep ='|',   
                                    encoding='utf-8-sig', 
                                    index=False)

#### CSV files for Innovation Intelligence (ONLY FVG local units and ONLY non-capital companies) – for integration into the I2 platform


In [51]:
# Filter out capital companies from the ANAGRAFICA dataset
tipo_società = ["SOCIETA' DI CAPITALE"]
anagrafica_soc_capitale = df_anagrafica["tipo_impresa"].isin(tipo_società)
df_anagrafica = df_anagrafica[~anagrafica_soc_capitale]

In [52]:
# Save the ANAGRAFICA file in the "Innovation Intelligence" version
# for platform integration after filtering out capital companies
file_risultati = data_dir + '\\' + 'iifvg_anagrafica_filtrato_noncap.csv'
cols_df = pd.read_excel('cols_dict.xlsx', sheet_name='anagrafica',
            usecols = ['nomi_colonne_corretti','ordine_iifvg']).dropna() 
cols_df.sort_values('ordine_iifvg', inplace = True)
cols_to_use = list(cols_df['nomi_colonne_corretti'])
df_anagrafica[cols_to_use].to_csv(  file_risultati, 
                                    sep ='|',   
                                    encoding='utf-8-sig', 
                                    index=False)

In [53]:
# Filter the CODICI dataset to keep only CF values present in the final ANAGRAFICA version
filtro_df_anagrafica = df_anagrafica[['cf']].copy()
df_codici = df_codici[df_codici['cf'].isin(filtro_df_anagrafica['cf'].values)]

In [54]:
# Save the CODICI file in the "Innovation Intelligence" version
# for platform integration after applying the final CF-based filter
file_risultati = data_dir + '\\' + 'iifvg_codici_filtrato_noncap.csv'
cols_df = pd.read_excel('cols_dict.xlsx', sheet_name='codici',
            usecols = ['nomi_colonne_corretti','ordine_iifvg']).dropna() 
cols_df.sort_values('ordine_iifvg', inplace = True)
cols_to_use = list(cols_df['nomi_colonne_corretti'])
df_codici[cols_to_use].to_csv(      file_risultati, 
                                    sep ='|',   
                                    encoding='utf-8-sig', 
                                    index=False)